# Plot example coarse behavioral labelling trace

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dandi/example-notebooks/blob/master/000055/BruntonLab/peterson21/Fig_coarse_labels.ipynb)

## Installing requirements

The cell below installs the Python packages needed to run this notebook and fetches helper files that are colocated with the notebook in the repo.

In [ ]:
# Expects Python 3.12 (Google Colab's default runtime as of 2026).
# Installs are scoped to the active interpreter via `uv pip install --system`,
# which is required because Colab runs the kernel outside a virtualenv.
!pip install -q uv
!uv pip install --system \
    "pynwb>=2.8,<3" \
    "hdmf<4" \
    "dandi>=0.60" \
    "h5py>=3.10" \
    "numpy>=1.24" \
    "pandas>=2.0" \
    "matplotlib>=3.7" \
    "seaborn>=0.13" \
    "scipy>=1.11" \
    "tqdm>=4.66" \
    "ipywidgets>=8" \
    "natsort>=8" \
    "ndx-events<0.3" \
    "nilearn>=0.10" \
    "nwbwidgets>=0.11" \
    "zarr<3" \
    "ipython_genutils"
!curl -sL -o plot_utils.py https://raw.githubusercontent.com/dandi/example-notebooks/master/000055/BruntonLab/peterson21/plot_utils.py

> **⚠️ Restart runtime after install**
>
> The install may upgrade packages already loaded in the kernel. Go to **Runtime → Restart session**, then **Run all cells below** (skip this install cell on re-run).

Replicates example coarse behavior labelling trace for one recording day. Note that the figure from the data paper combined the targeted (targeted=True) and untargeted (both first_val=True and first_val=False) behavior labels.

In [ ]:
%pip install -r requirements.txt

In [1]:
import natsort
from pynwb import NWBHDF5IO
from dandi.dandiapi import DandiAPIClient
import ndx_events

from plot_utils import prune_clabels, plot_clabels

## Set parameters

In [2]:
targ_tlims = [13, 17]  # targeted window to plot (in hours)
targeted = True  # plot targeted window (True) or whole day (False)
targ_label = 'Computer/phone'

## Load NWB data file

In [ ]:
with DandiAPIClient() as client:
    asset = clieconnt.get_dandiset("000055", "draft").get_asset_by_path(
        "sub-01/sub-01_ses-4_behavior+ecephys.nwb"
    )
    s3_path = asset.get_content_url(follow_redirects=1, strip_query=True)

with NWBHDF5IO(s3_path, mode='r', load_namespaces=True, driver='ros3') as io:
    nwb = io.read()
    clabels_orig = nwb.intervals['epochs'].to_dataframe()

## Select coarse labels based on user parameters

In [ ]:
label_col_d = {
    'Other activity': 0,
    'Computer/phone': 1,
    'Eat': 2,
    'TV': 3,
    'Talk': 4
}

clabels, uni_labs = prune_clabels(clabels_orig, targeted,
                                  targ_tlims, None,
                                  targ_label)

## Plot labels over time

In [ ]:
fig = plot_clabels(clabels, uni_labs, targeted, None, targ_tlims, targlab_colind=label_col_d[targ_label])


# # Save figure
# fig_sp = ''
# targ_label_out = '_'.join(targ_label.split(' '))
# targ_label_out = '_'.join(targ_label_out.split('/'))
# fig.savefig(fig_sp+targ_label_out+'_targeted_trace.png',
#             format='png',  transparent= True,dpi=150,
#             bbox_inches = 'tight', pad_inches = 0.01,
#             )